In [ ]:
from pathlib import Path
import random
import pickle
from tqdm.auto import tqdm

import pandas as pd
import numpy as np
from scipy.stats import ks_2samp, chi2_contingency, ttest_ind
import rpy2.robjects.numpy2ri
from rpy2.robjects.packages import importr
rpy2.robjects.numpy2ri.activate()

In [ ]:
SEED = 42
random.seed(SEED)
np.random.seed(42)

In [ ]:
real = pd.read_csv(
    Path.home()
    / "Research"
    / "Virtual-ALS-Patients"
    / "data"
    / "no_labels_no_birth_year"
    / "train.csv"
)
real.head()

In [ ]:
# filter out patients with no temporal data (all unknowns)

dfs = []

for _, pdf in real.groupby(by="REF"):
    if (pdf[[f"P{j}" for j in range(1, 13)]] == -1.0).all().all():
        continue
    dfs.append(pdf)
    
real = pd.concat(dfs, ignore_index=True)

In [ ]:
version = "2025-03-28 15:06:33"
fname = f"synthetic_dfs_n_samples=100_patientflow_{version}_seed=42.pkl"

with open(fname, "rb") as f:
    gen_dfs, categorical_feats_info = pickle.load(f)

In [ ]:
numerical_feats = ["Age_onset", "Height (m)", "Weight"]

categorical_feats = [
    "Gender",
    "Ethnicity",
    "NIV",
    "Tracheostomy",
    "PEG",
    "UMNvsLMN",
    "Onset",
    "Limb_O",
    "Limbs_Impairment",
    "Limbs_Side",
    "Weightloss_>10%",
    "ALS_familiar_history",
    "Ever_smoked",
    "Blood_hypertension",
    "Diabetes",
    "Dyslipidemia",
    "Thyroid",
    "Autoimmune",
    "SOD1 Mutation",
    "Stroke",
    "Cardiac_disease",
    "Primary_cancer",
    "C9orf72",
    "TARDBP mutation",
    "FUS mutation",
]

In [ ]:
p_threshold = 0.01

In [ ]:
ks_p_values = {}

print("KS-Tests")

for feat in numerical_feats:
    if feat not in ks_p_values:
        ks_p_values[feat] = []

    for gen_df in gen_dfs:
        df = pd.DataFrame({feat.lower(): list(map(lambda x: x[1][feat].iloc[0], real.groupby("REF"))), "type": "real"})
        df = pd.concat([df, pd.DataFrame({feat.lower(): list(map(lambda x: x[1][feat].iloc[0], gen_df.groupby("REF"))), "type": "fake"})], ignore_index=True)
        ks = ks_2samp(df[df["type"] == "real"][feat.lower()], df[df["type"] == "fake"][feat.lower()])
        ks_p_values[feat].append(ks.pvalue)

for feat in ks_p_values:
    p_values = np.array(ks_p_values[feat])
    print(f"Feature {feat}: {(p_values > p_threshold).sum()} / {len(p_values)}. p_values statistics: {p_values.mean()} +/- {p_values.std()}")

In [ ]:
t_test_p_values = {}
print("Welch T-Tests")

for feat in numerical_feats:
    if feat not in t_test_p_values:
        t_test_p_values[feat] = []
    
    for gen_df in gen_dfs:
        df = pd.DataFrame({feat.lower(): list(map(lambda x: x[1][feat].iloc[0], real.groupby("REF"))), "type": "real"})
        df = pd.concat([df, pd.DataFrame({feat.lower(): list(map(lambda x: x[1][feat].iloc[0], gen_df.groupby("REF"))), "type": "fake"})], ignore_index=True)
        t_unequal = ttest_ind(df[df["type"] == "real"][feat.lower()], df[df["type"] == "fake"][feat.lower()], equal_var=False)
        t_test_p_values[feat].append(t_unequal.pvalue)

for feat in t_test_p_values:
    p_values = np.array(t_test_p_values[feat])
    print(f"Feature {feat}: {(p_values > p_threshold).sum()} / {len(p_values)}. p_values statistics: {p_values.mean()} +/- {p_values.std()}")

In [ ]:
print("Three Sigma Rule Test")
sigma = 3

sigma_rule_tests = {}
    
for feat in numerical_feats:
    if feat not in sigma_rule_tests:
        sigma_rule_tests[feat] = []
    
    df = pd.DataFrame({feat.lower(): list(map(lambda x: x[1][feat].iloc[0], real.groupby("REF"))), "type": "real"})
    mean = df[df["type"] == "real"][feat.lower()].mean()
    std = df[df["type"] == "real"][feat.lower()].std()
    min_value = mean - sigma * std
    max_value = mean + sigma * std

    for gen_df in gen_dfs:
        df = pd.DataFrame({feat.lower(): list(map(lambda x: x[1][feat].iloc[0], gen_df.groupby("REF"))), "type": "fake"})
        sigma_rule_tests[feat].append(not ((df[feat.lower()] < min_value).any() and (df[feat.lower()] > max_value).any()))

for feat in sigma_rule_tests:
    sigma_passed_tests = sigma_rule_tests[feat]
    print(f"Feature {feat}: {sigma_passed_tests.count(True)} / {len(sigma_passed_tests)}")

In [ ]:
chi_square_p_values = {}

print("Chi-Square Tests")

for feat in categorical_feats:
    if feat not in chi_square_p_values:
        chi_square_p_values[feat] = []
    
    for gen_df in gen_dfs:
        df = pd.DataFrame({feat.lower(): list(map(lambda x: x[1][feat].iloc[0], real.groupby("REF"))), "type": "real"})
        df = pd.concat([df, pd.DataFrame({feat.lower(): list(map(lambda x: x[1][feat].iloc[0], gen_df.groupby("REF"))), "type": "fake"})], ignore_index=True)
        df[feat.lower()] = df[feat.lower()].apply(lambda x: x.lower() if isinstance(x, str) else x)
        
        counts_real = df[df["type"] == "real"][feat.lower()].value_counts().to_dict()
        counts_fake = df[df["type"] == "fake"][feat.lower()].value_counts().to_dict()
        table = np.zeros((2, len(counts_real.keys())), dtype=np.int32)
        
        for i, key in enumerate(counts_real.keys()):
            table[0, i] = counts_real[key]
            table[1, i] = counts_fake[key] if key in counts_fake else 0

        
        g, p, dof, expctd = chi2_contingency(table, lambda_='log-likelihood')
        chi_square_p_values[feat].append(p)

for feat in chi_square_p_values:
    p_values = np.array(chi_square_p_values[feat])
    print(f"Feature {feat}: {(p_values > p_threshold).sum()} / {len(p_values)}. p_values statistics: {p_values.mean()} +/- {p_values.std()}")

In [ ]:
print("Fisher's Exact Tests")

fisher_p_tests = {}

stats = importr('stats')

for feat in tqdm(categorical_feats):
    if feat not in fisher_p_tests:
        fisher_p_tests[feat] = []
    
    for gen_df in gen_dfs:
        df = pd.DataFrame({feat.lower(): list(map(lambda x: x[1][feat].iloc[0], real.groupby("REF"))), "type": "real"})
        df = pd.concat([df, pd.DataFrame({feat.lower(): list(map(lambda x: x[1][feat].iloc[0], gen_df.groupby("REF"))), "type": "fake"})], ignore_index=True)
        df[feat.lower()] = df[feat.lower()].apply(lambda x: x.lower() if isinstance(x, str) else x)
        
        counts_real = df[df["type"] == "real"][feat.lower()].value_counts().to_dict()
        # print(counts_real)
        counts_fake = df[df["type"] == "fake"][feat.lower()].value_counts().to_dict()
        # print(counts_fake)
        table = np.zeros((2, len(counts_real.keys())), dtype=np.int32)
        
        for i, key in enumerate(counts_real.keys()):
            table[0, i] = counts_real[key]
            table[1, i] = counts_fake[key] if key in counts_fake else 0

        p = stats.fisher_test(table, workspace=2e8)[0][0]
        fisher_p_tests[feat].append(p)

for feat in fisher_p_tests:
    p_values = np.array(fisher_p_tests[feat])
    print(f"Feature {feat}: {(p_values > p_threshold).sum()} / {len(p_values)}. p_values statistics: {p_values.mean()} +/- {p_values.std()}")